# Analyse des ouvertures Lichess

[▶ Ouvrir et exécuter dans Google Colab](https://colab.research.google.com/github/Hadrienfonrojet/ANALYSE-MARCHE-/blob/main/Analyse_data.ipynb)

GitHub affiche le notebook ; Colab fournit le calcul. Exécuter les cellules dans l’ordre. Les collectes utilisent l’API Lichess, les analyses restent corrélationnelles. Chaque exécution de la cellule « Nouvelle expérience » crée un dossier indépendant. Les fichiers Colab sont temporaires : télécharger l’archive finale avant de quitter.


In [ ]:
# 1. Installation des bibliothèques (dont boto3) et récupération du projet
import os, sys, subprocess
from pathlib import Path
if not Path('Pipeline_ouvertures.py').exists():
    target = Path('/content/ANALYSE-MARCHE-') if Path('/content').exists() else Path('ANALYSE-MARCHE-')
    if not target.exists():
        subprocess.run(['git', 'clone', 'https://github.com/Hadrienfonrojet/ANALYSE-MARCHE-.git', str(target)], check=True)
    os.chdir(target)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
print('Projet :', Path.cwd())


## 2. Paramètres
Le jeton est optionnel pour les parties publiques. Ne pas écrire de secret dans le notebook ni enregistrer une sortie contenant des identifiants. Les valeurs ci-dessous donnent un petit essai ; augmenter ensuite la taille.


In [ ]:
SEEDS = ['yuuki-asuna']  # Remplacer par vos comptes ; cinq graines conseillées
USERS_PER_SEED = 1
GAMES_PER_FORMAT = 5
RAW_FILE = None  # Ou un chemin vers un fichier games_enrichies_raw.parquet existant
os.environ['LICHESS_SEEDS'] = ','.join(SEEDS)


## 3. Nouvelle expérience
Cette cellule ne supprime rien. La relancer crée une nouvelle expérience.


In [ ]:
import config_projet as cfg
cfg.SEED_USERS = [s.lower().strip() for s in SEEDS]
run_dir = cfg.create_run()
print('Dossier de cette expérience :', run_dir)


## 4. Collecte via l’API ou réutilisation de données
Les requêtes sont séquentielles ; une limitation Lichess peut entraîner une attente.


In [ ]:
import pandas as pd
from Scripts_generation_data.Collecte_enrichie import discover_stratified, collect_games
if RAW_FILE is None:
    token = cfg.get_lichess_token()
    users = discover_stratified(cfg.SEED_USERS, token, USERS_PER_SEED)
    raw = collect_games(users, token, GAMES_PER_FORMAT)
else:
    raw = pd.read_parquet(RAW_FILE)
    raw.to_parquet(cfg.DATA_DIR / 'games_enrichies_raw.parquet', index=False)
print('Observations brutes :', len(raw))
raw.head()


## 5. Ouvertures et nettoyage
Le référentiel contient uniquement les ouvertures observées dans les réponses de l’API.


In [ ]:
from Scripts_generation_data.Creation_openings_complet import build_openings
from Nettoyage_donnees import prepare
openings = build_openings(raw)
clean = prepare(raw, openings)
display(openings.head())
display(clean[['user_id', 'opening_family', 'color', 'rating', 'result']].head())


## 6. Commandes d’analyse : tableaux descriptifs
Les cellules suivantes sont les commandes d’analyse exécutables. Le score vaut 1 pour une victoire, 0,5 pour un nul et 0 pour une défaite.


In [ ]:
description = clean.groupby(['opening_family', 'color'], observed=True).agg(
    parties=('game_id', 'size'), score_moyen=('score', 'mean'), taux_victoire=('win', 'mean')
).sort_values('parties', ascending=False)
display(description)


## 7. Tests du χ² et V de Cramér
`analyze` enregistre les tests et figures. Les régressions sont désactivées par défaut. `valid=False` signale des effectifs théoriques insuffisants. Les observations répétées par joueur et partie limitent l’indépendance : aucune interprétation causale.


In [ ]:
from Analyse_ouvertures import analyze
tests = analyze(clean)
display(tests)


## 8. Corrélations de Spearman
Associations entre variables numériques, sans interprétation causale. Les échantillons très petits ou constants donnent des corrélations indéfinies.


In [ ]:
import matplotlib.pyplot as plt
columns = ['rating', 'opponent_rating', 'elo_diff', 'experience_opening_prior', 'score']
corr = clean[columns].corr(method='spearman')
display(corr)
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(columns)), columns, rotation=45, ha='right')
ax.set_yticks(range(len(columns)), columns)
fig.colorbar(im, ax=ax, label='Spearman')
fig.tight_layout()
fig.savefig(cfg.RESULTS_DIR / 'correlations_spearman.png', dpi=150)
display(fig)
plt.close(fig)


## 9. Graphiques des ouvertures
Comparer les scores et les effectifs ; un score élevé sur peu de parties est fragile.


In [ ]:
summary = clean.groupby('opening_family').agg(parties=('score', 'size'), score=('score', 'mean')).nlargest(12, 'parties')
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
summary['parties'].plot.barh(ax=axes[0], title='Effectifs par famille')
summary['score'].plot.barh(ax=axes[1], title='Score moyen par famille', xlim=(0, 1))
fig.tight_layout()
fig.savefig(cfg.RESULTS_DIR / 'ouvertures_scores.png', dpi=150)
display(fig)
plt.close(fig)


## 10. Résultats et sauvegarde
Dans Colab, cette cellule télécharge les données et résultats de l’expérience. Elle n’envoie rien vers SSP Cloud. Les fichiers générés ne sont pas ajoutés automatiquement au dépôt GitHub.


In [ ]:
import shutil
print('Résultats :', cfg.RESULTS_DIR)
archive = shutil.make_archive(str(run_dir), 'zip', run_dir)
print('Archive :', archive)
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink
    display(FileLink(archive))
else:
    files.download(archive)


## 11. SSP Cloud — envoi optionnel
`boto3` est installé dans la première cellule. Renseigner les identifiants uniquement dans l’environnement privé. Passer `UPLOAD` à `True` pour envoyer cette expérience sous un préfixe unique.


In [ ]:
UPLOAD = False
if UPLOAD:
    from Stockage_SSPCloud import upload_outputs
    upload_outputs()
